In [8]:
import joblib
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
import os

# --- config (reuse your existing values) ---
from fl_supabase import _load_secrets
_S = _load_secrets()['supabase']   # credentials live in secrets.toml, never in the notebook
SUPABASE = {
    'host':     _S['host'],
    'port':     6543,
    'database': 'postgres',
    'user':     _S['user'],
    'password': _S['password'],
}
MODEL_PATH = r'C:\Users\215nt\PycharmProjects\FLStuff\FLHitting\simple_bat_speed_z_score_model.joblib'

features = ['launch_speed', 'height_inches', 'launch_angle', 'spray_angle', 'is_2_strikes']
FEATURE_DB_MAP = {
    'launch_speed':  'exit_speed',
    'launch_angle':  'angle',
    'spray_angle':   'direction',
    'height_inches': 'height_in',
}

def a_swing_mean(x):
    clean = x.dropna()
    if len(clean) < 5:
        return np.nan
    return clean[clean >= clean.quantile(0.10)].mean()

def get_engine():
    cfg = SUPABASE
    url = (f"postgresql+psycopg2://{cfg['user']}:{cfg['password']}"
           f"@{cfg['host']}:{cfg['port']}/{cfg['database']}")
    return create_engine(url, pool_pre_ping=True,
                         connect_args={'sslmode': 'require'})

# --- 1. pull 2026 Jackals BBE ---
q = text("""
    SELECT exit_speed, angle, direction, height_in, strikes,
           batter, batter_id, batter_team, date, pitch_call
    FROM pitches
    WHERE date::timestamp >= '2026-01-01'
      AND date::timestamp <= '2026-05-07'
      AND batter_team = 'Mississippi Mud Monsters'
      AND pitch_call = 'in_play'
""")
with get_engine().connect() as conn:
    df = pd.read_sql(q, conn)

df.columns = [c.lower().strip() for c in df.columns]
df = df.rename(columns={v: k for k, v in FEATURE_DB_MAP.items()})
df['is_2_strikes'] = (df['strikes'] == 2).astype(int)
df = df.dropna(subset=features).copy()
print(f'Pulled {len(df)} 2026 Jackals BBE across {df["batter"].nunique()} hitters')

# --- 2. z-score features using FL pull stats, then predict ---
model = joblib.load(MODEL_PATH)
X = pd.DataFrame(index=df.index)
for col in features:
    X[col] = (df[col] - df[col].mean()) / df[col].std()
df['pred_z_bat_speed'] = model.predict(X)

# --- 3. leaderboard: a_swing_mean per hitter, min 6 swings ---
lb = (df.groupby(['batter_id', 'batter'])
        .agg(swings=('pred_z_bat_speed', lambda x: x.dropna().shape[0]),
             avg_top90_pred_z=('pred_z_bat_speed', a_swing_mean))
        .reset_index())
lb = lb[lb['swings'] >= 6].copy()
lb = lb.sort_values('avg_top90_pred_z', ascending=False).reset_index(drop=True)
lb['rank'] = lb.index + 1

print(f'\n--- 2026 New Jersey Jackals Bat Speed Leaderboard ---')
print(f'(top 90% trimmed mean of pred_z_bat_speed, min 6 swings)\n')
print(lb[['rank', 'batter', 'swings', 'avg_top90_pred_z']].to_string(index=False))

Pulled 225 2026 Jackals BBE across 22 hitters

--- 2026 New Jersey Jackals Bat Speed Leaderboard ---
(top 90% trimmed mean of pred_z_bat_speed, min 6 swings)

 rank             batter  swings  avg_top90_pred_z
    1       Holman, Jack      16          0.462216
    2 Updegrave, Charlie       7          0.381719
    3    Wallace, Landon       9          0.279322
    4       Holt, Travis      17          0.225817
    5    Bocchino, Peter      13          0.209956
    6          Fritz, AJ      17          0.207394
    7  Nicklaus, Jackson      13          0.170125
    8         Bane, Gray      11          0.084567
    9       Semo, Andrew      10          0.074018
   10       Booker, Kyle      20          0.003548
   11  De La Rosa, Samil      13         -0.002849
   12    Schield, Slater      13         -0.006447
   13      Reed, Vantrel       6         -0.079499
   14  Skinner, Brayland      22         -0.100646
   15      Zitella, Jake      14         -0.135133
   16       Furr, Kasten 